In [ ]:
import numpy as np
import scipy as sp
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.pyplot as plt
import pandas as pd
import time

In [ ]:
df = pd.read_csv('/Users/livialamers/Documents/GitHub/Spatial-data-science/data/datapoging2.csv',sep = ';', keep_default_na = True)


In [4]:
df.head()

,ID,WijkenEnBuurten,Gemeentenaam_1,SoortRegio_2,Codering_3,IndelingswijzigingWijkenEnBuurten_4,AfstandTotHuisartsenpraktijk_5,Binnen1Km_6,Binnen3Km_7,Binnen5Km_8,...,Binnen5Km_105,Binnen10Km_106,Binnen20Km_107,AfstandTotSauna_108,AfstandTotZonnebank_109,AfstandTotAttractie_110,Binnen10Km_111,Binnen20Km_112,Binnen50Km_113,AfstandTotBrandweerkazerne_114
0,1170,GM0363,Amsterdam,Gemeente,GM0363,3,0.6,3.0,28.3,67.4,...,5.5,11.7,18.2,2.5,2.0,3.2,5.3,13.8,61.4,2.1
1,1171,WK0363AA,Amsterdam,Wijk,WK0363AA,2,0.5,3.8,32.2,96.0,...,10.8,13.0,20.8,1.5,0.8,3.3,6.9,14.0,62.2,1.9
2,1172,BU0363AA01,Amsterdam,Buurt,BU0363AA01,2,0.8,3.8,30.8,87.8,...,11.0,13.0,20.7,1.9,0.4,3.7,6.6,15.0,63.0,1.7
3,1173,BU0363AA02,Amsterdam,Buurt,BU0363AA02,2,0.8,2.3,29.5,87.5,...,10.6,13.0,20.8,1.8,0.7,3.6,6.7,14.9,62.5,2.0
4,1174,BU0363AA03,Amsterdam,Buurt,BU0363AA03,2,0.2,2.0,28.8,92.9,...,10.7,13.0,20.4,1.5,1.3,3.2,7.0,10.7,61.7,2.4


In [5]:
df.isna().sum()

ID                                 0
WijkenEnBuurten                    0
Gemeentenaam_1                     0
SoortRegio_2                       0
Codering_3                         0
                                  ..
AfstandTotAttractie_110           41
Binnen10Km_111                    41
Binnen20Km_112                    41
Binnen50Km_113                    41
AfstandTotBrandweerkazerne_114    41
Length: 98, dtype: int64

In [6]:
cols_to_drop = [col for col in df.columns if col.startswith("Binnen")]
#cols_to_drop

In [7]:
df = df.drop(columns=cols_to_drop)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 38 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   ID                                      628 non-null    int64  
 1   WijkenEnBuurten                         628 non-null    object 
 2   Gemeentenaam_1                          628 non-null    object 
 3   SoortRegio_2                            628 non-null    object 
 4   Codering_3                              628 non-null    object 
 5   IndelingswijzigingWijkenEnBuurten_4     628 non-null    int64  
 6   AfstandTotHuisartsenpraktijk_5          587 non-null    float64
 7   AfstandTotHuisartsenpost_9              587 non-null    float64
 8   AfstandTotApotheek_10                   587 non-null    float64
 9   AfstandTotZiekenhuis_11                 587 non-null    float64
 10  AfstandTotZiekenhuis_15                 587 non-null    float6

In [2]:
df.

SyntaxError: invalid syntax (1625957470.py, line 1)

In [2]:
print ('YORAN SEE MEE PLEASE')

YORAN SEE MEE PLEASE


Hier met de grouping spelen: 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# =========================
# 0) LOAD YOUR DATA
# =========================
# Pas dit pad aan naar jouw opgeslagen csv
# bv: total_final_df = pd.read_csv("data/total_final_df.csv")
total_final_df = pd.read_csv("data/total_final_df.csv")

print("Loaded:", total_final_df.shape)
print(total_final_df.columns[:10])

# =========================
# 1) YOUR HYPOTHESIS GROUPS
# =========================
groups = {
    "nightlife": [
        "AfstandTotCafeED_36",
        "AfstandTotCafetariaED_40",
        "AfstandTotRestaurant_44",
        "AfstandTotPoppodium_103",
        "AfstandTotBioscoop_104",
        "AfstandTotHotelED_48",
    ],
    "mobility": [
        "AfstandTotOpritHoofdverkeersweg_89",
        "AfstandTotTreinstationsTotaal_90",
        "AfstandTotBelangrijkOverstapstation_91",
    ],
    "retail_daily": [
        "AfstandTotGroteSupermarkt_24",
        "AfstandTotOvDagelLevensmiddelen_28",
        "AfstandTotWarenhuis_32",
    ],
    "education_youth": [
        "AfstandTotKinderdagverblijf_52",
        "AfstandTotBuitenschoolseOpvang_56",
        "AfstandTotSchool_60",
        "AfstandTotSchool_64",
        "AfstandTotSchool_68",
        "AfstandTotSchool_72",
    ],
    "culture_sport_public": [
        "AfstandTotBibliotheek_92",
        "AfstandTotZwembad_93",
        "AfstandTotKunstijsbaan_94",
        "AfstandTotMuseum_95",
        "AfstandTotPodiumkunstenTotaal_99",
        "AfstandTotAttractie_110",
    ],
    "emergency_health": [
        "AfstandTotBrandweerkazerne_114",
        "AfstandTotHuisartsenpraktijk_5",
        "AfstandTotHuisartsenpost_9",
        "AfstandTotApotheek_10",
        "AfstandTotZiekenhuis_11",
        "AfstandTotZiekenhuis_15",
    ],
}

# reverse map: facility -> hypothese label
hyp_map = {}
for gname, cols in groups.items():
    for c in cols:
        hyp_map[c] = gname

# =========================
# 2) SELECT FACILITY COLUMNS
# =========================
facility_cols = [c for c in total_final_df.columns if c.startswith("AfstandTot")]

# (OPTIONEEL) drop "sauna/zonnebank" als je die niet wil laten meespelen:
# facility_cols = [c for c in facility_cols if ("Sauna" not in c and "Zonnebank" not in c)]

print("Number of facility vars:", len(facility_cols))

df_fac = total_final_df[facility_cols].copy()

# missing handling: simplest is drop rows with any missing in facility vars
df_fac = df_fac.dropna()
print("Rows after dropna on facilities:", df_fac.shape[0])

# =========================
# 3) VARIABLE CLUSTERING (OFFICIAL)
#    We cluster *variables*, so transpose: (n_vars, n_rows)
# =========================
X_var = df_fac.to_numpy().T  # shape = (n_vars, n_wijken)

# ---- choose number of clusters via silhouette (correlation distance) ----
k_candidates = range(3, 9)  # try 3..8 clusters
scores = []

for k in k_candidates:
    model = AgglomerativeClustering(
        n_clusters=k,
        metric="correlation",   # 1 - corr
        linkage="average"
    )
    labels = model.fit_predict(X_var)
    score = silhouette_score(X_var, labels, metric="correlation")
    scores.append((k, score))

scores_df = pd.DataFrame(scores, columns=["k", "silhouette_corr"])
print(scores_df)

best_k = scores_df.sort_values("silhouette_corr", ascending=False).iloc[0]["k"]
best_score = scores_df["silhouette_corr"].max()
print(f"\nBest k = {int(best_k)} (silhouette={best_score:.3f})")

# ---- fit final clustering ----
cluster_model = AgglomerativeClustering(
    n_clusters=int(best_k),
    metric="correlation",
    linkage="average"
)
cluster_labels = cluster_model.fit_predict(X_var)

var_clusters = pd.DataFrame({
    "facility_var": facility_cols,
    "alg_cluster": cluster_labels
}).sort_values(["alg_cluster", "facility_var"])

# add hypothese label
var_clusters["hyp_group"] = var_clusters["facility_var"].map(hyp_map).fillna("not_in_hypothesis")

display(var_clusters)

# =========================
# 4) COMPARE CLUSTERS vs HYPOTHESIS
# =========================
ct = pd.crosstab(var_clusters["alg_cluster"], var_clusters["hyp_group"])
display(ct)

dominant = ct.idxmax(axis=1)
dominant_share = (ct.max(axis=1) / ct.sum(axis=1)).round(2)

summary = pd.DataFrame({
    "dominant_hyp_group": dominant,
    "dominant_share": dominant_share,
    "n_vars_in_cluster": ct.sum(axis=1)
}) doc
display(summary)

# =========================
# 5) MAKE CLUSTER FEATURES (MEAN DISTANCE PER CLUSTER)
# =========================
df_cluster_features = pd.DataFrame(index=df_fac.index)

for cl in sorted(var_clusters["alg_cluster"].unique()):
    cols_in_cl = var_clusters.loc[var_clusters["alg_cluster"] == cl, "facility_var"].tolist()
    df_cluster_features[f"mean_dist_cluster_{cl}"] = df_fac[cols_in_cl].mean(axis=1)

display(df_cluster_features.head())
print("Cluster feature columns:", df_cluster_features.columns.tolist())

# (OPTIONEEL) Merge terug met originele df (zelfde index als df_fac na dropna)
df_with_clusters = total_final_df.loc[df_fac.index].copy()
df_with_clusters = pd.concat([df_with_clusters, df_cluster_features], axis=1)

print("\nMerged df shape:", df_with_clusters.shape)


Loaded: (110, 56)
Index(['wijkcode', 'wijknaam', 'aantal_lantaarns', 'Totaal misdrijven',
       'wijkcode_clean', 'AfstandTotHuisartsenpraktijk_5',
       'AfstandTotHuisartsenpost_9', 'AfstandTotApotheek_10',
       'AfstandTotZiekenhuis_11', 'AfstandTotZiekenhuis_15'],
      dtype='object')
Number of facility vars: 32
Rows after dropna on facilities: 110
   k  silhouette_corr
0  3         0.231953
1  4         0.338687
2  5         0.321724
3  6         0.349336
4  7         0.336001
5  8         0.302578

Best k = 6 (silhouette=0.349)


,facility_var,alg_cluster,hyp_group
30,AfstandTotAttractie_110,0,culture_sport_public
27,AfstandTotBioscoop_104,0,nightlife
1,AfstandTotHuisartsenpost_9,0,emergency_health
26,AfstandTotPoppodium_103,0,nightlife
4,AfstandTotZiekenhuis_15,0,emergency_health
29,AfstandTotZonnebank_109,0,not_in_hypothesis
2,AfstandTotApotheek_10,1,emergency_health
21,AfstandTotBibliotheek_92,1,culture_sport_public
13,AfstandTotBuitenschoolseOpvang_56,1,education_youth
9,AfstandTotCafetariaED_40,1,nightlife


hyp_group,culture_sport_public,education_youth,emergency_health,mobility,nightlife,not_in_hypothesis,retail_daily
alg_cluster,,,,,,,
0,1,0,2,0,2,1,0
1,1,6,2,0,3,0,3
2,3,0,2,0,1,1,0
3,0,0,0,1,0,0,0
4,1,0,0,0,0,0,0
5,0,0,0,2,0,0,0


,dominant_hyp_group,dominant_share,n_vars_in_cluster
alg_cluster,,,
0,emergency_health,0.33,6
1,education_youth,0.40,15
2,culture_sport_public,0.43,7
3,mobility,1.00,1
4,culture_sport_public,1.00,1
5,mobility,1.00,2


,mean_dist_cluster_0,mean_dist_cluster_1,mean_dist_cluster_2,mean_dist_cluster_3,mean_dist_cluster_4,mean_dist_cluster_5
0,2.766667,0.560000,1.285714,3.4,6.5,1.6
1,2.316667,0.553333,0.814286,3.2,6.8,2.3
2,2.383333,0.626667,0.757143,3.6,6.3,1.8
3,2.400000,0.740000,0.857143,4.0,6.0,1.5
4,2.266667,0.746667,0.942857,4.4,5.4,1.1


Cluster feature columns: ['mean_dist_cluster_0', 'mean_dist_cluster_1', 'mean_dist_cluster_2', 'mean_dist_cluster_3', 'mean_dist_cluster_4', 'mean_dist_cluster_5']

Merged df shape: (110, 62)


In [2]:
import numpy as np
import pandas as pd

target_raw = "Misdrijven_per_1000_inwoners"

# log target (aanrader voor stabiele relaties)
df_eval = total_final_df.copy()
df_eval["log_crime_rate"] = np.log1p(df_eval[target_raw])

# jouw handmatige mean features (bestaan al in total_final_df)
manual_mean_cols = [
    "mean_dist_nightlife",
    "mean_dist_mobility",
    "mean_dist_retail_daily",
    "mean_dist_education_youth",
    "mean_dist_culture_sport_public",
    "mean_dist_emergency_health",
]

# algorithmic cluster features (die heb je gemaakt als df_cluster_features of in df_with_clusters)
# stel dat je df_with_clusters heet en die kolommen bevat:
cluster_cols = [c for c in df_with_clusters.columns if c.startswith("mean_dist_cluster_")]

# maak één evaluatie df met dezelfde rijen (belangrijk: vergelijk eerlijk)
common_idx = df_eval.index.intersection(df_with_clusters.index)
df_eval = df_eval.loc[common_idx].copy()
df_alg = df_with_clusters.loc[common_idx, cluster_cols].copy()

# combineer alles
df_all = pd.concat([df_eval[["log_crime_rate"] + manual_mean_cols], df_alg], axis=1).dropna()

def corr_table(df, y, cols):
    out = []
    for c in cols:
        out.append({
            "feature": c,
            "spearman": df[c].corr(df[y], method="spearman"),
            "pearson":  df[c].corr(df[y], method="pearson"),
        })
    return pd.DataFrame(out).sort_values("spearman", key=lambda s: s.abs(), ascending=False)

corr_manual = corr_table(df_all, "log_crime_rate", manual_mean_cols)
corr_alg    = corr_table(df_all, "log_crime_rate", cluster_cols)

print("=== Manual groups (mean_dist_*) correlations ===")
display(corr_manual)

print("=== Algorithmic clusters (mean_dist_cluster_*) correlations ===")
display(corr_alg)


=== Manual groups (mean_dist_*) correlations ===


,feature,spearman,pearson
0,mean_dist_nightlife,-0.452376,-0.199287
4,mean_dist_culture_sport_public,-0.378670,-0.098520
5,mean_dist_emergency_health,-0.238119,-0.037193
2,mean_dist_retail_daily,-0.232717,0.108910
1,mean_dist_mobility,-0.163742,-0.200195
3,mean_dist_education_youth,-0.078092,0.351648


=== Algorithmic clusters (mean_dist_cluster_*) correlations ===


,feature,spearman,pearson
2,mean_dist_cluster_2,-0.379509,-0.093321
0,mean_dist_cluster_0,-0.351270,-0.186999
5,mean_dist_cluster_5,-0.330938,-0.279250
3,mean_dist_cluster_3,0.270158,0.224296
4,mean_dist_cluster_4,-0.231384,-0.023250
1,mean_dist_cluster_1,-0.219704,0.219458


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import make_scorer, mean_squared_error, mean_absolute_error

# target + filtering (zelfde regels als jullie modelling)
df_base = total_final_df.copy()
df_base = df_base[df_base["inwoners"] >= 50].copy()
df_base["log_crime_rate"] = np.log1p(df_base["Misdrijven_per_1000_inwoners"])

# voeg algorithmic cluster features toe (zelfde index!)
# df_with_clusters is de merged df uit clustering code
common_idx = df_base.index.intersection(df_with_clusters.index)
df_base = df_base.loc[common_idx].copy()

cluster_cols = [c for c in df_with_clusters.columns if c.startswith("mean_dist_cluster_")]
df_base = df_base.join(df_with_clusters.loc[common_idx, cluster_cols])

# maak twee feature sets:
X_manual = ["Trees_per_km2", "Lights_per_km2"] + manual_mean_cols
X_alg    = ["Trees_per_km2", "Lights_per_km2"] + cluster_cols

# dropna eerlijk voor beide (zodat je niet per set andere wijken gebruikt)
df_manual = df_base[["log_crime_rate"] + X_manual].dropna().copy()
df_alg    = df_base[["log_crime_rate"] + X_alg].dropna().copy()

# als je EXACT dezelfde rows wilt voor eerlijke vergelijking:
common_idx2 = df_manual.index.intersection(df_alg.index)
df_manual = df_manual.loc[common_idx2]
df_alg    = df_alg.loc[common_idx2]

y_manual = df_manual["log_crime_rate"].values
y_alg    = df_alg["log_crime_rate"].values

X_manual_mat = df_manual[X_manual].values
X_alg_mat    = df_alg[X_alg].values

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scorer = make_scorer(lambda yt, yp: np.sqrt(mean_squared_error(yt, yp)), greater_is_better=False)
mae_scorer  = make_scorer(mean_absolute_error, greater_is_better=False)

def eval_set(X, y):
    scores = cross_validate(
        pipe, X, y, cv=cv,
        scoring={"r2":"r2", "rmse": rmse_scorer, "mae": mae_scorer},
        return_train_score=False
    )
    return {
        "R2_mean": scores["test_r2"].mean(),
        "R2_std":  scores["test_r2"].std(),
        "RMSE_mean": -scores["test_rmse"].mean(),
        "RMSE_std":  scores["test_rmse"].std(),
        "MAE_mean": -scores["test_mae"].mean(),
        "MAE_std":  scores["test_mae"].std(),
        "n": len(y)
    }

res_manual = eval_set(X_manual_mat, y_manual)
res_alg    = eval_set(X_alg_mat, y_alg)

results = pd.DataFrame([res_manual, res_alg], index=["Manual groups", "Algorithmic clusters"]).round(3)
display(results)


,R2_mean,R2_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std,n
Manual groups,0.203,0.169,0.599,0.098,0.464,0.079,109
Algorithmic clusters,0.171,0.179,0.603,0.040,0.444,0.032,109


NameError: name 'df_model' is not defined